In [8]:
from utils import get_dataset_lines
import math

# Inferring clusters from a tree

HierarchicalClustering, whose pseudocode is shown below, progressively generates $n$ different partitions of the underlying data into clusters, all represented by a tree in which each node is labeled by a cluster of genes. The first partition has $n$ single-element clusters represented by the leaves of the tree, with each element forming its own cluster. The second partition merges the two “closest” clusters into a single cluster consisting of two elements. In general, the $i$-th partition merges the two closest clusters from the $(i - 1)$-th partition and has $n - i + 1$ clusters. We hope this algorithm looks familiar — it is UPGMA (from the chapter on evolutionary tree construction) in disguise.

```plaintext
HierarchicalClustering(D, n)
    Clusters ← n single-element clusters labeled 1, ... , n 
      construct a graph T with n isolated nodes labeled by single elements 1, ... , n 
    while there is more than one cluster 
        find the two closest clusters Ci and Cj 
        merge Ci and Cj into a new cluster Cnew with |Ci| + |Cj| elements
        add a new node labeled by cluster Cnew to T
        connect node Cnew to Ci and Cj by directed edges
        remove the rows and columns of D corresponding to Ci and Cj
        remove Ci and Cj from Clusters
        add a row/column to D for Cnew by computing D(Cnew, C) for each C in Clusters 
        add Cnew to Clusters 
    assign root in T as a node with no incoming edges
    return T
```

**Code Challenge**: Implement HierarchicalClustering.

**Input**: An integer $n$, followed by an $n \times n$ distance matrix.

**Output**: The result of applying HierarchicalClustering to this distance matrix (using $D_{avg}$), with each newly created cluster listed on each line.

**Sample Input**:

```
7
0.00 0.74 0.85 0.54 0.83 0.92 0.89
0.74 0.00 1.59 1.35 1.20 1.48 1.55
0.85 1.59 0.00 0.63 1.13 0.69 0.73
0.54 1.35 0.63 0.00 0.66 0.43 0.88
0.83 1.20 1.13 0.66 0.00 0.72 0.55
0.92 1.48 0.69 0.43 0.72 0.00 0.80
0.89 1.55 0.73 0.88 0.55 0.80 0.00
```

**Sample Output**:

```
4 6
5 7
3 4 6
1 2
5 7 3 4 6
1 2 5 7 3 4 6
```

In [2]:
def parse_hierarchical_clustering_input(lines):
    lines = [l.strip() for l in lines if l.strip()]
    if not lines:
        return 0, []
    n = int(lines[0])
    matrix = []
    for line in lines[1:]:
        matrix.append(list(map(float, line.split())))
    return n, matrix

def HierarchicalClustering(n, matrix):
    # Clusters map: cluster_id -> list of members (1-based indices)
    # Ideally cluster_id 0 corresponds to member [1], 1 to [2], etc.
    clusters = {i: [i+1] for i in range(n)}
    
    # Distances map: frozenset({id1, id2}) -> distance
    distances = {}
    for i in range(n):
        for j in range(i + 1, n):
            distances[frozenset((i, j))] = matrix[i][j]
            
    next_cluster_id = n
    result_clusters = []
    
    while len(clusters) > 1:
        current_ids = sorted(list(clusters.keys()))
        
        min_dist = float('inf')
        closest_pair = None
        
        # Find closest pair of clusters
        for i in range(len(current_ids)):
            id1 = current_ids[i]
            for j in range(i + 1, len(current_ids)):
                id2 = current_ids[j]
                d = distances.get(frozenset((id1, id2)), float('inf'))
                if d < min_dist:
                    min_dist = d
                    closest_pair = (id1, id2)
                    
        if closest_pair is None:
            break
            
        c1_id, c2_id = closest_pair
        
        # Merge clusters
        # Note: c1_id < c2_id because of the loop structure and sorting
        new_cluster_members = clusters[c1_id] + clusters[c2_id]
        clusters[next_cluster_id] = new_cluster_members
        
        # Store result line
        result_clusters.append(" ".join(map(str, new_cluster_members)))
        
        # Update distances using Davg formula (average linkage)
        # D(New, Other) = (|C1|*D(C1, Other) + |C2|*D(C2, Other)) / (|C1| + |C2|)
        
        len_c1 = len(clusters[c1_id])
        len_c2 = len(clusters[c2_id])
        len_new = len_c1 + len_c2
        
        for other_id in current_ids:
            if other_id != c1_id and other_id != c2_id:
                d1 = distances[frozenset((c1_id, other_id))]
                d2 = distances[frozenset((c2_id, other_id))]
                
                new_dist = (len_c1 * d1 + len_c2 * d2) / len_new
                distances[frozenset((next_cluster_id, other_id))] = new_dist
        
        # Remove merged clusters
        del clusters[c1_id]
        del clusters[c2_id]
        
        next_cluster_id += 1
        
    return result_clusters

In [3]:
### Sample Input Execution
sample_input = """
7
0.00 0.74 0.85 0.54 0.83 0.92 0.89
0.74 0.00 1.59 1.35 1.20 1.48 1.55
0.85 1.59 0.00 0.63 1.13 0.69 0.73
0.54 1.35 0.63 0.00 0.66 0.43 0.88
0.83 1.20 1.13 0.66 0.00 0.72 0.55
0.92 1.48 0.69 0.43 0.72 0.00 0.80
0.89 1.55 0.73 0.88 0.55 0.80 0.00
""".strip().split('\n')

n_sample, matrix_sample = parse_hierarchical_clustering_input(sample_input)
result = HierarchicalClustering(n_sample, matrix_sample)

for line in result:
    print(line)

### Verification
expected_output = [
    "4 6",
    "5 7",
    "3 4 6",
    "1 2",
    "5 7 3 4 6",
    "1 2 5 7 3 4 6"
]

assert result == expected_output, f"Expected {expected_output}, but got {result}"
print("Sample test passed!")

4 6
5 7
3 4 6
1 2
5 7 3 4 6
1 2 5 7 3 4 6
Sample test passed!


In [5]:
### Dataset Test
dataset_filename = 'dataset_30177_7.txt' 

try:
    lines = get_dataset_lines(dataset_filename)
    if lines:
        n, matrix = parse_hierarchical_clustering_input(lines)
        result = HierarchicalClustering(n, matrix)
        
        for line in result:
            print(line)
            
except FileNotFoundError:
    print(f"File {dataset_filename} not found. Please download the dataset.")
except Exception as e:
    print(f"An error occurred: {e}")

6 14
4 10
9 20
5 12
2 8
7 18
13 5 12
3 11
9 20 2 8
16 4 10
15 6 14
1 9 20 2 8
7 18 15 6 14
13 5 12 16 4 10
17 19
3 11 7 18 15 6 14
13 5 12 16 4 10 17 19
1 9 20 2 8 13 5 12 16 4 10 17 19
3 11 7 18 15 6 14 1 9 20 2 8 13 5 12 16 4 10 17 19


In [ ]:
# Coursera Quiz Questions

# Question 2
# Given the following Data and Centers, compute HiddenMatrix_2,5 (i.e., the responsibility of the second center for the fifth datapoint) using the Newtonian inverse-square law. Give your answer to three decimal places.

Data =  [(2,8), (2,5), (6,9), (7,5), (5,2)]
Centers = [(3,5), (5,4)]

# Calculate Inverse Square Distances
# Data point 5 is at index 4 (0-based)
point = Data[4]
inv_sq_dists = []
for center in Centers:
    dist = math.dist(point, center)
    inv_sq_dists.append(1.0 / (dist ** 2))

# Responsibility is proportional to inverse square distance
# HM_2,5 corresponds to index 1 (2nd center)
total_inv_sq = sum(inv_sq_dists)
responsibility_2_5 = inv_sq_dists[1] / total_inv_sq

print(f"Q2 Answer: {responsibility_2_5:.3f}")

# Question 3
# Say we have the following Data and HiddenMatrix:

Data_Q3 = [(2,8), (2,5), (6,9), (7,5), (5,2)]

HiddenMatrix = [
    [0.5, 0.3, 0.8, 0.4, 0.9],
    [0.5, 0.7, 0.2, 0.6, 0.1]
]

# Compute the weighted center of gravity corresponding to the second row of HiddenMatrix. Please enter your coordinates (x, y) in the form
# x y
# rounded to three decimal places

# Row index 1 for second row
row_index = 1
weights = HiddenMatrix[row_index]

sum_wx = 0.0
sum_wy = 0.0
sum_w = sum(weights)

for i, w in enumerate(weights):
    px, py = Data_Q3[i]
    sum_wx += w * px
    sum_wy += w * py

cg_x = sum_wx / sum_w
cg_y = sum_wy / sum_w

print(f"Q3 Answer: {cg_x:.3f} {cg_y:.3f}")

Q2 Answer: 0.765
Q3 Answer: 3.952 5.952
